# Notebook version of `mlp_from_scratch.py`


This notebook mirrors the Python script and keeps output-interpretation notes in the script comments.
Run the code cell below to reproduce outputs.

In [ ]:
# Output Guide:
# - Prints training start text and final `Accuracy`.
# - Accuracy close to 1.0 indicates the MLP learned the nonlinear decision boundary.
# - A clearly above-chance value (>0.80 for this toy task) is typically a correct outcome.
"""
Multilayer Perceptron (MLP) — NumPy From Scratch
=================================================
Architecture : Input(2) → Hidden(12, ReLU) → Output(2, Softmax)
Task         : Binary classification on a simple nonlinear dataset
Training     : Batch gradient descent with cross-entropy loss
"""

import numpy as np


# ── Reproducibility ──────────────────────────────────────────────────────────
rng = np.random.default_rng(42)


# ── Dataset ──────────────────────────────────────────────────────────────────
def make_dataset(n: int = 240):
    """Generate a simple 2-D nonlinear binary classification dataset."""
    X = rng.normal(size=(n, 2))
    # Label is 1 when x0*x1 + 0.25*x0 > 0
    y = ((X[:, 0] * X[:, 1] + 0.25 * X[:, 0]) > 0).astype(int)
    Y = np.eye(2)[y]          # one-hot encode labels
    return X, y, Y


# ── Activation functions ──────────────────────────────────────────────────────
def relu(z: np.ndarray) -> np.ndarray:
    """Rectified Linear Unit: max(0, z)."""
    return np.maximum(0, z)


def softmax(z: np.ndarray) -> np.ndarray:
    """Numerically stable row-wise softmax."""
    exp_z = np.exp(z - z.max(axis=1, keepdims=True))
    return exp_z / exp_z.sum(axis=1, keepdims=True)


# ── Weight initialisation ─────────────────────────────────────────────────────
def init_weights(input_dim: int = 2, hidden_dim: int = 12, output_dim: int = 2):
    """Xavier-ish normal initialisation."""
    W1 = rng.normal(scale=0.4, size=(input_dim, hidden_dim))
    b1 = np.zeros(hidden_dim)
    W2 = rng.normal(scale=0.4, size=(hidden_dim, output_dim))
    b2 = np.zeros(output_dim)
    return W1, b1, W2, b2


# ── Forward pass ──────────────────────────────────────────────────────────────
def forward(X, W1, b1, W2, b2):
    """
    Compute hidden activations and output probabilities.

    Returns
    -------
    h : ReLU activations from the hidden layer  (N, hidden_dim)
    p : Softmax probabilities from output layer  (N, output_dim)
    """
    h = relu(X @ W1 + b1)       # hidden pre-activation → ReLU
    p = softmax(h @ W2 + b2)    # output pre-activation → softmax
    return h, p


# ── Training loop ─────────────────────────────────────────────────────────────
def train(X, Y, W1, b1, W2, b2, lr: float = 0.1, epochs: int = 600):
    """
    Batch gradient descent minimising cross-entropy loss.

    Gradient derivation (cross-entropy + softmax combined):
        dL/d(pre-softmax) = (p - Y) / N          ← 'delta output'
    Then standard chain rule back through W2 → ReLU → W1.
    """
    n = len(X)

    for epoch in range(epochs):
        # ── Forward ──────────────────────────────────────────────────────
        h, p = forward(X, W1, b1, W2, b2)

        # ── Output-layer gradient ─────────────────────────────────────────
        d_out = (p - Y) / n                  # shape: (N, output_dim)

        # ── Gradients for W2, b2 ─────────────────────────────────────────
        dW2 = h.T @ d_out                    # (hidden, output)
        db2 = d_out.sum(axis=0)              # (output,)

        # ── Back-propagate through ReLU ───────────────────────────────────
        d_hidden = (d_out @ W2.T) * (h > 0) # ReLU derivative is 0 where h≤0

        # ── Gradients for W1, b1 ─────────────────────────────────────────
        dW1 = X.T @ d_hidden                 # (input, hidden)
        db1 = d_hidden.sum(axis=0)           # (hidden,)

        # ── Gradient descent update ───────────────────────────────────────
        W1 -= lr * dW1
        b1 -= lr * db1
        W2 -= lr * dW2
        b2 -= lr * db2

    return W1, b1, W2, b2


# ── Evaluation ────────────────────────────────────────────────────────────────
def evaluate(X, y, W1, b1, W2, b2):
    """Compute classification accuracy."""
    _, p = forward(X, W1, b1, W2, b2)
    predictions = p.argmax(axis=1)
    accuracy = (predictions == y).mean()
    return accuracy


# ── Main ──────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    X, y, Y = make_dataset()
    W1, b1, W2, b2 = init_weights()

    print("Training MLP from scratch …")
    W1, b1, W2, b2 = train(X, Y, W1, b1, W2, b2)

    acc = evaluate(X, y, W1, b1, W2, b2)
    print(f"Accuracy : {acc:.3f}")